<a href="https://colab.research.google.com/github/Adigozalovh/DeepLearning/blob/main/Cityscapes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader

In [2]:
class Conv3(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()
    self.calc = nn.Sequential(
        nn.Conv2d(n_features, out_channel, 3, 1 , 1),
        nn.ReLU(),
        nn.Conv2d(out_channel, out_channel, 3, 1, 1),
        nn.ReLU()
    )

  def forward(self, X):
    return self.calc(X)

In [3]:
class UpConv(nn.Module):
    def __init__(self, n_features, out_channel, kernel_size=2):
        super().__init__()
        self.upconv = nn.ConvTranspose2d(n_features, out_channel, kernel_size=kernel_size, stride=2)

    def forward(self, X):
        return self.upconv(X)

In [4]:
class UNet(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()

    self.max_pool = nn.MaxPool2d(2, 2)

    self.DownConv1 = Conv3(n_features, 64)
    self.DownConv2 = Conv3(64, 128)
    self.DownConv3 = Conv3(128, 256)
    self.DownConv4 = Conv3(256, 512)

    self.Bottleneck = Conv3(512, 1024)

    self.UpConv1 = UpConv(1024, 512)
    self.UpConv2 = UpConv(512, 256)
    self.UpConv3 = UpConv(256, 128)
    self.UpConv4 = UpConv(128, 64)

    self.Expansion1 = Conv3(1024, 512)
    self.Expansion2 = Conv3(512, 256)
    self.Expansion3 = Conv3(256, 128)
    self.Expansion4 = Conv3(128, 64)

    self.output = nn.Conv2d(64, out_channel, 1, stride = 1)

  def forward(self, X):
    X = self.DownConv1(X)
    skip1 = X
    X = self.max_pool(X)
    X = self.DownConv2(X)
    skip2 = X
    X = self.max_pool(X)
    X = self.DownConv3(X)
    skip3 =X
    X = self.max_pool(X)
    X = self.DownConv4(X)
    skip4 = X
    X = self.max_pool(X)
    X = self.Bottleneck(X)

    X = self.UpConv1(X)
    X = torch.cat([X, skip4], dim = 1)
    X = self.Expansion1(X)
    X = self.UpConv2(X)
    X = torch.cat([X, skip3], dim = 1)
    X = self.Expansion2(X)
    X = self.UpConv3(X)
    X = torch.cat([X, skip2], dim = 1)
    X = self.Expansion3(X)
    X = self.UpConv4(X)
    X = torch.cat([X, skip1], dim = 1)
    X = self.Expansion4(X)

    X = self.output(X)

    return X

In [5]:
def calculate_dice_score(preds, labels, eps=1e-8):
  proba = torch.sigmoid(preds)
  preds - (proba > 0.5).float()

  preds = preds.view(-1)
  labels = labels.view(-1)

  intersection = (preds * labels).sum()
  dice_score = (2.0 * intersection + eps) / (preds.sum() + eps)

  return dice_score

In [6]:
def eval(model, criterion, valid_loader, device):
  valid_losses = 0.
  valid_dice_scores = 0.
  model.eval()

  with torch.no_grad():
    for image, mask in valid_loader:
      image, mask = image.to(device), mask.to(device)
      preds = model(image)
      loss = criterion(preds, mask)
      dice_score = calculate_dice_score(preds, mask)
      valid_losses += loss.item()
      valid_dice_scores += dice_score.item()

    avg_loss = valid_losses / len(valid_loader)
    avg_dice_score = valid_dice_scores / len(valid_loader)

  return avg_loss, avg_dice_score

In [7]:
if torch.cuda.is_available():
  device='cuda'
elif torch.backends.mps.is_available():
  device='mps'
else:
  device='cpu'

In [8]:
def train(model, criterion, optimizer, train_loader,
          valid_loader, scheduler, n_epochs, device):
  history = {
      'train_loss':[],
      'train_metric':[],
      'valid_loss':[],
      'valid_metric':[],
      'lr':[]
  }

  best_dice_score = 0.
  for epoch in range(n_epochs):
    train_losses = 0.
    train_dice_scores = 0.
    model.train()
    current_lr = optimizer.param_groups[0]['lr']
    for image, mask in train_loader:
      image, mask = image.to(device), mask.to(device)
      with torch.cuda.amp.autocast(enabled=(device=='cuda')):
        preds = model(image)
        loss = criterion(preds, mask)
        dice_score = calculate_dice_score(preds, mask)
        train_losses += loss.item()
        train_dice_scores += dice_score.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    avg_train_loss = train_losses / len(train_loader)
    avg_dice_score = train_dice_scores / len(train_loader)
    avg_valid_loss, avg_valid_dice_score = eval(model, criterion,
                                                valid_loader, device)

    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
      scheduler.step(avg_valid_loss)
    else:
      scheduler.step()

    if avg_dice_score > best_dice_score:
      best_dice_score = avg_dice_score

    history['train_loss'].append(avg_train_loss)
    history['train_metric'].append(avg_dice_score)
    history['valid_loss'].append(avg_valid_loss)
    history['valid_metric'].append(avg_valid_dice_score)
    history['lr'].append(current_lr)

    print({f'Epoch: {epoch+1}/{n_epochs}| Train Loss: {avg_train_loss:^5}| Train Dice Score: {avg_dice_score:^5}| Val Loss: {avg_valid_loss:^5}| Val Dice Score: {avg_valid_dice_score:^5}| LR: {current_lr}'})

  return history

In [9]:
!curl -L -o cityscapes.zip https://www.kaggle.com/api/v1/datasets/download/shuvoalok/cityscapes

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  199M  100  199M    0     0   122M      0  0:00:01  0:00:01 --:--:--  174M


In [10]:
!unzip /content/cityscapes.zip

Streaming output truncated to the last 5000 lines.
  inflating: train/img/train2754.png  
  inflating: train/img/train2755.png  
  inflating: train/img/train2756.png  
  inflating: train/img/train2757.png  
  inflating: train/img/train2758.png  
  inflating: train/img/train2759.png  
  inflating: train/img/train276.png  
  inflating: train/img/train2760.png  
  inflating: train/img/train2761.png  
  inflating: train/img/train2762.png  
  inflating: train/img/train2763.png  
  inflating: train/img/train2764.png  
  inflating: train/img/train2765.png  
  inflating: train/img/train2766.png  
  inflating: train/img/train2767.png  
  inflating: train/img/train2768.png  
  inflating: train/img/train2769.png  
  inflating: train/img/train277.png  
  inflating: train/img/train2770.png  
  inflating: train/img/train2771.png  
  inflating: train/img/train2772.png  
  inflating: train/img/train2773.png  
  inflating: train/img/train2774.png  
  inflating: train/img/train2775.png  
  inflating: tr

In [11]:
import os
import glob
from PIL import Image

In [12]:
class SegmentationDataset(Dataset):
  def __init__(self, image_path, mask_path):
    self.image_files = sorted(glob.glob(os.path.join(image_path, '*.png')))
    self.mask_files = sorted(glob.glob(os.path.join(mask_path, '*.png')))

    self.transform = transforms.Compose([
        transforms.ToTensor()
    ])

  def __len__(self):
    return len(self.image_files)

  def __getitem__(self, idx):
    transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
])
    image = Image.open(self.image_files[idx]).convert('L')
    mask = Image.open(self.mask_files[idx]).convert('L')

    return self.transform(image), self.transform(mask)

In [13]:
train_data = SegmentationDataset(image_path = '/content/train/img',
                                 mask_path='/content/train/label')
val_data = SegmentationDataset(image_path = '/content/val/img',
                                 mask_path='/content/val/label')

In [14]:
train_loader, valid_loader = DataLoader(train_data, batch_size=1, shuffle=True), DataLoader(val_data, batch_size=1)

In [15]:
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        inputs = torch.sigmoid(inputs)

        bce = F.binary_cross_entropy(inputs, targets, reduction='mean')

        return bce

In [16]:
model = UNet(1, 1)
model.to(device)
criterion = nn.BCEWithLogitsLoss()
learning_rate = 1e-3
n_epochs=10
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

history = train(model, criterion, optimizer, train_loader, valid_loader, scheduler, n_epochs, device)

/tmp/ipykernel_616/874928795.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=='cuda')):


{'Epoch: 1/10| Train Loss: 0.6515773424180615| Train Dice Score: 0.661168936310696| Val Loss: 0.6362832931280136| Val Dice Score: 0.6438669254183769| LR: 0.001'}
{'Epoch: 2/10| Train Loss: 0.6373925018711251| Train Dice Score: 0.6487069278103965| Val Loss: 0.6352040727138519| Val Dice Score: 0.6443415002822876| LR: 0.001'}
{'Epoch: 3/10| Train Loss: 0.6367758639319604| Train Dice Score: 0.6446674128039545| Val Loss: 0.6341325870752335| Val Dice Score: 0.636572635948658| LR: 0.001'}
{'Epoch: 4/10| Train Loss: 0.6361328565573492| Train Dice Score: 0.6409671097543059| Val Loss: 0.6336642415523529| Val Dice Score: 0.6303126767277718| LR: 0.001'}
{'Epoch: 5/10| Train Loss: 0.6356757924135994| Train Dice Score: 0.6387804046398451| Val Loss: 0.6340354726314544| Val Dice Score: 0.6319343838095665| LR: 0.001'}
{'Epoch: 6/10| Train Loss: 0.6353420487371814| Train Dice Score: 0.6368275539314046| Val Loss: 0.6330007612705231| Val Dice Score: 0.6291676042675972| LR: 0.001'}
{'Epoch: 7/10| Train Los